In [2]:
# # 01 — NFHS-5 Mapping

# Purpose: inspect the available NFHS-5 indicator CSV and create the first mapping layer for the synthetic-risk pipeline.

# ## Frozen pipeline
# 1. NFHS-5 aggregated indicators → prevalence/reference values
# 2. DA-WI → published risk-factor relationships and weights
# 3. Risk-factor mapping → common synthetic schema
# 4. Seed dataset generation
# 5. CTGAN generation
# 6. Model training and on-device quantization

# **Important:** this CSV is an aggregated NFHS-5 indicator table, not respondent-level microdata. It is therefore used as an empirical reference, not directly as CTGAN training rows.

In [3]:
import pandas as pd
import os
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)

df_nfhs = pd.read_csv('nfhs_survey.csv')

print('Shape:', df_nfhs.shape)
df_nfhs.head()

Shape: (4819, 9)


,S.No.,indicator code,Indicators,sub indicators,NFHS-5 (2019-20),Unnamed: 5,Unnamed: 6,NFHS-4 2015-16,STATE/UT
0,NaN,NaN,NaN,NaN,Urban,Rural,Total,Total,NaN
1,1.0,1.0,Population and Household Profile,Female population age 6 years and above who ever attended school (%),86.5,81.8,83.5,84.7,Andaman & Nicobar Islands
2,2.0,2.0,Population and Household Profile,Population below age 15 years (%),22.7,19.7,20.8,23.9,Andaman & Nicobar Islands
3,3.0,3.0,Population and Household Profile,"Sex ratio of the total population (females per 1,000 males)",1023,929,963,977,Andaman & Nicobar Islands
4,4.0,4.0,Population and Household Profile,"Sex ratio at birth for children born in the last five years (females per 1,000 males)",941,891,914,859,Andaman & Nicobar Islands


In [4]:
print('Columns:')
for col in df_nfhs.columns:
    print(repr(col))
            
print('\nData types:')
print(df_nfhs.dtypes)

Columns:
'S.No.'
'indicator code'
'Indicators'
' sub indicators'
'NFHS-5 (2019-20)'
'Unnamed: 5'
'Unnamed: 6'
'NFHS-4 2015-16'
'STATE/UT'

Data types:
S.No.               float64
indicator code      float64
Indicators           object
 sub indicators      object
NFHS-5 (2019-20)     object
Unnamed: 5           object
Unnamed: 6           object
NFHS-4 2015-16       object
STATE/UT             object
dtype: object


In [5]:
# Remove accidental whitespace from column names only.
df_nfhs.columns = df_nfhs.columns.str.strip()

print(df_nfhs.columns.tolist())
print('\nRows:', len(df_nfhs))

['S.No.', 'indicator code', 'Indicators', 'sub indicators', 'NFHS-5 (2019-20)', 'Unnamed: 5', 'Unnamed: 6', 'NFHS-4 2015-16', 'STATE/UT']

Rows: 4819


In [6]:
# ## 1. Find violence and relevant contextual indicators

# We search the indicator and sub-indicator text rather than assuming exact NFHS variable names.

In [7]:
keywords = [
    'violence', 'violent', 'physical', 'sexual', 'emotional',
    'spousal', 'pregnancy', 'injury', 'husband', 'partner',
    'control', 'help', 'alcohol', 'tobacco'
]

text_cols = ['Indicators', 'sub indicators']

mask = pd.Series(False, index=df_nfhs.index)
for col in text_cols:
    mask |= df_nfhs[col].astype(str).str.contains(
        '|'.join(keywords), case=False, na=False
    )

nfhs_relevant = df_nfhs.loc[mask].copy()

print('Relevant rows:', nfhs_relevant.shape[0])
display(nfhs_relevant[['Indicators', 'sub indicators', 'NFHS-5 (2019-20)', 'STATE/UT']])

Relevant rows: 407


,Indicators,sub indicators,NFHS-5 (2019-20),STATE/UT
98,Blood Sugar Level among Adults (age 15 years and above) - Women,Blood sugar level - high or very high (>140 mg/dl) or taking medicine to control blood sugar level (%) - Women,19.6,Andaman & Nicobar Islands
101,Blood Sugar Level among Adults (age 15 years and above) - Men,Blood sugar level - high or very high (>140 mg/dl) or taking medicine to control blood sugar level (%) - Men,19.4,Andaman & Nicobar Islands
104,Hypertension among Adults (age 15 years and above) - Women,Elevated blood pressure (Systolic ≥140 mm of Hg and/or Diastolic ≥90 mm of Hg) or taking medicine to control blood p...,23.4,Andaman & Nicobar Islands
107,Hypertension among Adults (age 15 years and above) - Men,Elevated blood pressure (Systolic ≥140 mm of Hg and/or Diastolic ≥90 mm of Hg) or taking medicine to control blood p...,28.2,Andaman & Nicobar Islands
122,Gender Based Violence (age 18-49 years),Ever-married women age 18-49 years who have ever experienced spousal violence (%),23.2,Andaman & Nicobar Islands
...,...,...,...,...
4814,Gender Based Violence (age 18-49 years),Young women age 18-29 years who experienced sexual violence by age (%),0.0,Puducherry
4815,Tobacco Use and Alcohol Consumption among Adults (age 15 years and above),Women age 15 years and above who use any kind of tobacco (%),1.2,Puducherry
4816,Tobacco Use and Alcohol Consumption among Adults (age 15 years and above),Men age 15 years and above who use any kind of tobacco (%),13.8,Puducherry
4817,Tobacco Use and Alcohol Consumption among Adults (age 15 years and above),Women age 15 years and above who consume alcohol (%),0.1,Puducherry


In [8]:
# Specifically inspect violence-related indicators.
violence_mask = (
    df_nfhs['sub indicators'].astype(str).str.contains(
        'spousal violence|physical violence|sexual violence|emotional violence|pregnancy',
        case=False, na=False
    )
)

violence_reference = df_nfhs.loc[
    violence_mask,
    ['Indicators', 'sub indicators', 'NFHS-5 (2019-20)', 'STATE/UT']
].copy()

display(violence_reference)

,Indicators,sub indicators,NFHS-5 (2019-20),STATE/UT
122,Gender Based Violence (age 18-49 years),Ever-married women age 18-49 years who have ever experienced spousal violence (%),23.2,Andaman & Nicobar Islands
123,Gender Based Violence (age 18-49 years),Ever-married women age 18-49 years who have experienced physical violence during any pregnancy (%),(0.0),Andaman & Nicobar Islands
124,Gender Based Violence (age 18-49 years),Young women age 18-29 years who experienced sexual violence by age (%),1.4,Andaman & Nicobar Islands
253,Gender Based Violence (age 18-49 years),Ever-married women age 18-49 years who have ever experienced spousal violence (%),28.8,Andhra Pradesh
254,Gender Based Violence (age 18-49 years),Ever-married women age 18-49 years who have experienced physical violence during any pregnancy (%),3.5,Andhra Pradesh
...,...,...,...,...
4682,Gender Based Violence (age 18-49 years),Ever-married women age 18-49 years who have experienced physical violence during any pregnancy (%),3.7,NCT Delhi
4683,Gender Based Violence (age 18-49 years),Young women age 18-29 years who experienced sexual violence by age (%),1.6,NCT Delhi
4812,Gender Based Violence (age 18-49 years),Ever-married women age 18-49 years who have ever experienced spousal violence (%),29.8,Puducherry
4813,Gender Based Violence (age 18-49 years),Ever-married women age 18-49 years who have experienced physical violence during any pregnancy (%),1.3,Puducherry


In [9]:
# ## 2. Create the NFHS reference layer

# The values below are reference/prevalence information. They are not individual-level observations and will not be passed directly into CTGAN.

In [10]:
# Save the relevant NFHS reference table for the next notebook.
os.makedirs('../data/processed', exist_ok=True)

nfhs_relevant.to_csv(
    '../data/processed/nfhs_relevant_indicators.csv',
    index=False
)

violence_reference.to_csv(
    '../data/processed/nfhs_violence_reference.csv',
    index=False
)

print('Saved NFHS reference tables.')

Saved NFHS reference tables.


In [11]:
# ## 3. Frozen mapping schema

# This is the common risk-factor layer used by the following notebooks. A factor is marked `NFHS` only when this aggregated CSV provides relevant evidence; otherwise its grounding comes from DA-WI or application-specific safety logic.

# | Common factor | NFHS reference | DA-WI | App-specific |
# |---|---:|---:|---:|
# | violence_escalation | Yes/related | Yes | No |
# | threat_to_kill | Not directly available | Yes | No |
# | violent_jealousy | Not directly available | Yes | No |
# | recent_separation | Not directly available | Yes | No |
# | lethal_weapon | Not directly available | Yes | No |
# | avoids_arrest | Not directly available | Yes | No |
# | strangulation | Not directly available | Yes | No |
# | illegal_drug_use | Not directly established by this table | Yes | No |
# | problem_drinking | Alcohol context available | Yes | No |
# | violence_during_pregnancy | Yes | Yes | No |
# | partner_capable_of_killing | Not directly available | Yes | No |
# | suicide_threat_attempt | Not directly available | Yes | No |
# | withholds_necessities | Not directly available | Yes | No |
# | intimidating_behavior | Not directly available | Yes | No |
# | social_isolation | Not directly available | Yes | No |
# | healthcare_neglect | Not directly available | Yes | No |
# | threat_for_leaving | Not directly available | Yes | No |
# | hides_abuse | Not directly available | Yes | No |
# | family_supports_abuse | Not directly available | Yes | No |
# | safe_now | No | No | Yes |
# | perpetrator_present | No | No | Yes |
# | can_leave_safely | No | No | Yes |
# | medical_help | No | No | Yes |
# | contact_requested | No | No | Yes |

In [12]:
# ## Output of Notebook 01

# - `data/processed/nfhs_relevant_indicators.csv`
# - `data/processed/nfhs_violence_reference.csv`

# Next: **02_risk_relationships.ipynb** — encode the published DA-WI factors, RRRs and weights into the frozen reference table.

In [13]:
# ============================================================
# FINAL OUTPUT FOR NOTEBOOK 01
# Run this entire cell and send me the output
# ============================================================

print("=" * 100)
print("1. NFHS RELEVANT INDICATORS")
print("=" * 100)

display(
    nfhs_relevant[
        ["Indicators", "sub indicators", "NFHS-5 (2019-20)", "STATE/UT"]
    ].drop_duplicates()
)


print("\n" + "=" * 100)
print("2. NFHS VIOLENCE REFERENCE")
print("=" * 100)

display(
    violence_reference[
        ["Indicators", "sub indicators", "NFHS-5 (2019-20)", "STATE/UT"]
    ]
)


print("\n" + "=" * 100)
print("3. SHAPE OF EXTRACTED TABLES")
print("=" * 100)

print("Full NFHS dataset:", df_nfhs.shape)
print("Relevant indicators:", nfhs_relevant.shape)
print("Violence reference:", violence_reference.shape)

1. NFHS RELEVANT INDICATORS


,Indicators,sub indicators,NFHS-5 (2019-20),STATE/UT
98,Blood Sugar Level among Adults (age 15 years and above) - Women,Blood sugar level - high or very high (>140 mg/dl) or taking medicine to control blood sugar level (%) - Women,19.6,Andaman & Nicobar Islands
101,Blood Sugar Level among Adults (age 15 years and above) - Men,Blood sugar level - high or very high (>140 mg/dl) or taking medicine to control blood sugar level (%) - Men,19.4,Andaman & Nicobar Islands
104,Hypertension among Adults (age 15 years and above) - Women,Elevated blood pressure (Systolic ≥140 mm of Hg and/or Diastolic ≥90 mm of Hg) or taking medicine to control blood p...,23.4,Andaman & Nicobar Islands
107,Hypertension among Adults (age 15 years and above) - Men,Elevated blood pressure (Systolic ≥140 mm of Hg and/or Diastolic ≥90 mm of Hg) or taking medicine to control blood p...,28.2,Andaman & Nicobar Islands
122,Gender Based Violence (age 18-49 years),Ever-married women age 18-49 years who have ever experienced spousal violence (%),23.2,Andaman & Nicobar Islands
...,...,...,...,...
4814,Gender Based Violence (age 18-49 years),Young women age 18-29 years who experienced sexual violence by age (%),0.0,Puducherry
4815,Tobacco Use and Alcohol Consumption among Adults (age 15 years and above),Women age 15 years and above who use any kind of tobacco (%),1.2,Puducherry
4816,Tobacco Use and Alcohol Consumption among Adults (age 15 years and above),Men age 15 years and above who use any kind of tobacco (%),13.8,Puducherry
4817,Tobacco Use and Alcohol Consumption among Adults (age 15 years and above),Women age 15 years and above who consume alcohol (%),0.1,Puducherry



2. NFHS VIOLENCE REFERENCE


,Indicators,sub indicators,NFHS-5 (2019-20),STATE/UT
122,Gender Based Violence (age 18-49 years),Ever-married women age 18-49 years who have ever experienced spousal violence (%),23.2,Andaman & Nicobar Islands
123,Gender Based Violence (age 18-49 years),Ever-married women age 18-49 years who have experienced physical violence during any pregnancy (%),(0.0),Andaman & Nicobar Islands
124,Gender Based Violence (age 18-49 years),Young women age 18-29 years who experienced sexual violence by age (%),1.4,Andaman & Nicobar Islands
253,Gender Based Violence (age 18-49 years),Ever-married women age 18-49 years who have ever experienced spousal violence (%),28.8,Andhra Pradesh
254,Gender Based Violence (age 18-49 years),Ever-married women age 18-49 years who have experienced physical violence during any pregnancy (%),3.5,Andhra Pradesh
...,...,...,...,...
4682,Gender Based Violence (age 18-49 years),Ever-married women age 18-49 years who have experienced physical violence during any pregnancy (%),3.7,NCT Delhi
4683,Gender Based Violence (age 18-49 years),Young women age 18-29 years who experienced sexual violence by age (%),1.6,NCT Delhi
4812,Gender Based Violence (age 18-49 years),Ever-married women age 18-49 years who have ever experienced spousal violence (%),29.8,Puducherry
4813,Gender Based Violence (age 18-49 years),Ever-married women age 18-49 years who have experienced physical violence during any pregnancy (%),1.3,Puducherry



3. SHAPE OF EXTRACTED TABLES
Full NFHS dataset: (4819, 9)
Relevant indicators: (407, 9)
Violence reference: (111, 4)
